# CS 3892 / 5892 — Session 21 · Cslib: machine-checked foundations, and the gap

**Tuesday, November 10, 2026 (a guest lecture).** **Cslib** is Lean's library for computer science,
built on Mathlib. At the course pin, v4.33.0, it proves results from sessions 12 and 14: the
**Hennessy–Milner theorem**, that **trace equivalence is coarser than bisimilarity**, the **Büchi
intersection** construction, that some ω-regular language has **no deterministic Büchi automaton**,
and **closure under complement**. It has **no LTL**, no LTL → Büchi translation, and no emptiness
decision procedure.

> **Run this locally or in a Codespace, not in Colab.** It needs Mathlib: about **7.5 GB** of
> disk and a few minutes the first time. Nothing graded needs it: every output is also in the
> slides and in `sessions/cs3892-2026-11-10-cslib-machine-checked-foundations/README.md`.

## Setup

The second cell installs Lean if needed, fetches Mathlib's cache and builds the project.

In [ ]:
# --- Setup: find the repo (clone on Colab), define helpers
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-11-10-cslib-machine-checked-foundations"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

PYD = ROOT / "sessions" / SESSION / "python"
LD  = ROOT / "sessions" / SESSION / "lean"
FIG = ROOT / "sessions" / SESSION / "figures"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True, cwd=str(path.parent))
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

from IPython.display import SVG, display
print("ready — helpers: show(path), run(path)")

In [ ]:
# --- Lean v4.33.0 (HW4's installer: elan into ~/.elan, then Lean), then MATHLIB'S CACHE and the
# build: `lake exe cache get` (about 1.5 minutes, 7.5 GB under lean/.lake) and `lake build`, which
# compiles only the Cslib modules this session imports. Needs ~10 GB free; not for Colab.
out = run(PYD / "lean_cli.py")
assert "version 4.33.0" in out

## 1. Session 14's vending machines, told apart by one HML formula

`vend` holds both machines. `⟨coin⟩¬⟨tea⟩⊤` holds at `idleE` and not at `idleL`, and Cslib's
Hennessy–Milner theorem turns that into a proof that the two are not bisimilar.

In [ ]:
display(SVG(filename=str(FIG / "vend_lts.svg")))
out = run(PYD / "01_vending_hml.py")
assert "not bisimilar" in out

## 2. The Hennessy–Milner theorem, as Cslib states it

Same HML formulas iff bisimilar, for image-finite systems; and bisimilar is strictly finer than
same traces.

In [ ]:
out = run(PYD / "02_hennessy_milner.py")
assert "image-finite" in out

## 3. Büchi automata: intersection, deterministic vs nondeterministic, complement

Three theorems from session 12's material, with what each rests on.

In [ ]:
display(SVG(filename=str(FIG / "nba_fg_n.svg")))
out = run(PYD / "03_buchi.py")
assert "standard axioms" in out

## 4. The whole project builds

`lake exe cache get`, then `lake build`: no `sorry`, no warning.

In [ ]:
show(LD / "lakefile.toml")
out = run(PYD / "04_whole_library.py")
assert "Build completed successfully" in out or "builds with no sorry" in out

## Try it yourself

Open `lean/S21/` in VS Code with the Lean 4 extension (it reads `lean-toolchain`).

- In `Vending.lean`, find a formula that holds at `idleL` and not at `idleE`. (Hint: `[coin]`, the
  box, is `¬⟨coin⟩¬`.) Prove both halves.
- Add a third machine that is bisimilar to LATE but has more states. Can you prove it, with
  `IsBisimulation`?
- Search Cslib for `LTL`. What would you need to add to check an LTL property of `vend`?